# Perbandingan Strategi Penanganan Kondisi Operasi pada LSTM Autoencoder
**C-MAPSS FD002 dan FD004** — Strategi A (normalisasi per *regime* operasi) vs Strategi B (normalisasi global + *operational setting* sebagai input), dengan *baseline* Isolation Forest.

**Cara pakai**
1. Jalankan sel 1 dan catat versi pustaka untuk bagian 2.2 makalah.
2. Jalankan sel 2, lalu unggah `train_FD002.txt` dan `train_FD004.txt`.
3. Sel 3 berisi konfigurasi. `QUICK = True` hanya uji coba cepat (1 seed, 5 epoch). Setelah berjalan lancar, ubah ke `QUICK = False` untuk hasil makalah (3 seed, hingga 40 epoch).
4. Jalankan sel 4 sampai 7. Hasil (Tabel 4, Gambar 4, CSV) terunduh sebagai `hasil_eksperimen.zip`.

Runtime: CPU sudah cukup (GPU tidak wajib). Mode penuh dapat memakan beberapa puluh menit.

In [ ]:
# Sel 1 - Impor pustaka dan cek versi (salin versi ini ke bagian 2.2 makalah)
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("TensorFlow  :", tf.__version__)
print("GPU         :", tf.config.list_physical_devices("GPU"))

## 2. Unggah data

In [ ]:
# Sel 2 - Unggah data: pilih train_FD002.txt dan train_FD004.txt dari komputer
DATA = {"FD002": "train_FD002.txt", "FD004": "train_FD004.txt"}
missing = [f for f in DATA.values() if not os.path.exists(f)]
if missing:
    from google.colab import files
    up = files.upload()           # pilih kedua file sekaligus
    for name in list(up):         # Colab menambah " (1)" bila nama sudah ada
        base = name.split(" (")[0]
        if base != name and base in DATA.values():
            os.replace(name, base)
for f in DATA.values():
    assert os.path.exists(f), f"File {f} belum ada, jalankan ulang sel ini dan unggah."
print("Data siap:", list(DATA.values()))

## 3. Konfigurasi dan fungsi

In [ ]:
# Sel 3 - Konfigurasi dan fungsi
# Ubah QUICK = False untuk hasil akhir makalah (3 seed, hingga 40 epoch).
QUICK = True

SPLIT_SEED = 42                       # pembagian mesin 80/20 dan K-means (tetap)
SEEDS = [42] if QUICK else [42, 43, 44]   # seed untuk inisialisasi model / Isolation Forest
EPOCHS = 5 if QUICK else 40
WINDOW = 30
HEALTHY_FRAC = 0.30
LATE_FRAC = 0.20
K_SIGMA = 3.0
PERSIST = 3
N_REGIMES = 6
SENSORS = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]

COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
OPS = ["op1", "op2", "op3"]
SENS = [f"s{i}" for i in SENSORS]
OUT = "hasil"
os.makedirs(OUT, exist_ok=True)


def load(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COLS)
    df["max_cycle"] = df.groupby("unit")["cycle"].transform("max")
    df["frac"] = df["cycle"] / df["max_cycle"]
    return df


def split_engines(df, val_frac=0.2):
    units = df["unit"].unique()
    rng = np.random.default_rng(SPLIT_SEED)
    rng.shuffle(units)
    n_val = int(len(units) * val_frac)
    return df[df["unit"].isin(units[n_val:])].copy(), df[df["unit"].isin(units[:n_val])].copy()


def make_windows(x, w=WINDOW):
    if len(x) < w:
        return np.empty((0, w, x.shape[1])), np.empty((0,), dtype=int)
    idx = np.arange(len(x) - w + 1)
    return np.stack([x[i:i + w] for i in idx]), idx + w - 1


class StrategyA:
    """Normalisasi sensor per regime operasi (K-means pada operational setting)."""
    name = "A"

    def fit(self, train):
        self.ops_sc = StandardScaler().fit(train[OPS])
        self.km = KMeans(N_REGIMES, n_init=10, random_state=SPLIT_SEED).fit(self.ops_sc.transform(train[OPS]))
        healthy = train[train["frac"] <= HEALTHY_FRAC].copy()
        healthy["reg"] = self.km.predict(self.ops_sc.transform(healthy[OPS]))
        self.fallback = StandardScaler().fit(healthy[SENS])
        self.scalers = {r: StandardScaler().fit(g[SENS]) for r, g in healthy.groupby("reg") if len(g) > 1}
        return self

    def transform(self, df):
        reg = self.km.predict(self.ops_sc.transform(df[OPS]))
        out = self.fallback.transform(df[SENS]).copy()
        for r, sc in self.scalers.items():
            m = reg == r
            if m.any():
                out[m] = sc.transform(df.loc[m, SENS])
        return out


class StrategyB:
    """Normalisasi global; tiga operational setting ikut sebagai fitur input."""
    name = "B"

    def fit(self, train):
        healthy = train[train["frac"] <= HEALTHY_FRAC]
        self.sc = StandardScaler().fit(healthy[SENS + OPS])
        return self

    def transform(self, df):
        return self.sc.transform(df[SENS + OPS])


def build_lstm_ae(n_feat, w=WINDOW):
    inp = keras.Input(shape=(w, n_feat))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.LSTM(32)(x)                      # bottleneck
    x = layers.RepeatVector(w)(x)
    x = layers.LSTM(32, return_sequences=True)(x)
    x = layers.LSTM(64, return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_feat))(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m


def make_and_fit(Xt, Xv, seed, epochs):
    keras.utils.set_random_seed(seed)
    model = build_lstm_ae(Xt.shape[2])
    h = model.fit(Xt, Xt, validation_data=(Xv, Xv), epochs=epochs, batch_size=128, verbose=0,
                  callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    return model, h.history


def recon_error(model, wins):
    pred = model.predict(wins, batch_size=512, verbose=0)
    return np.mean((wins - pred) ** 2, axis=(1, 2))


def first_persistent_alarm(flags, persist=PERSIST):
    run = 0
    for i, f in enumerate(flags):
        run = run + 1 if f else 0
        if run >= persist:
            return i                      # alarm dibunyikan saat run ke-p tercapai
    return None


def metrics_row(unit, g, flag, frac, cyc):
    healthy = frac <= HEALTHY_FRAC
    late = frac >= (1 - LATE_FRAC)
    a = first_persistent_alarm(flag)
    first_cycle = cyc[a] if a is not None else np.nan
    return {
        "unit": unit,
        "far": flag[healthy].mean() if healthy.any() else np.nan,
        "recall": flag[late].mean() if late.any() else np.nan,
        "first_alarm_cycle": first_cycle,
        "lead_time": (g["cycle"].max() - first_cycle) if a is not None else np.nan,
        "detected": a is not None,
        "early_false": bool(a is not None and frac[a] <= HEALTHY_FRAC),
    }


def eval_lstm(model, thr, prep, val, keep=None):
    rows, curve = [], None
    for unit, g in val.groupby("unit"):
        g = g.sort_values("cycle")
        wins, last = make_windows(prep.transform(g))
        if len(wins) == 0:
            continue
        err = recon_error(model, wins)
        flag = err > thr
        rows.append(metrics_row(unit, g, flag, g["frac"].values[last], g["cycle"].values[last]))
        if keep is not None and unit == keep:
            curve = dict(cycle=g["cycle"].values[last], err=err, thr=thr,
                         alarm=rows[-1]["first_alarm_cycle"], T=g["cycle"].max())
    return pd.DataFrame(rows), curve


def eval_iforest(iso, prep, val):
    rows = []
    for unit, g in val.groupby("unit"):
        g = g.sort_values("cycle")
        flag = iso.predict(prep.transform(g)) == -1
        rows.append(metrics_row(unit, g, flag, g["frac"].values, g["cycle"].values))
    return pd.DataFrame(rows)


def run_lstm(prep_cls, train, val, seed, keep=None):
    prep = prep_cls().fit(train)
    X = []
    for _, g in train.groupby("unit"):
        g = g.sort_values("cycle")
        w, _ = make_windows(prep.transform(g[g["frac"] <= HEALTHY_FRAC]))
        if len(w):
            X.append(w)
    X = np.concatenate(X)
    n_val = int(0.15 * len(X))
    idx = np.random.default_rng(seed).permutation(len(X))
    Xv, Xt = X[idx[:n_val]], X[idx[n_val:]]
    model, hist = make_and_fit(Xt, Xv, seed, EPOCHS)
    ev = recon_error(model, Xv)
    thr = ev.mean() + K_SIGMA * ev.std()
    res, curve = eval_lstm(model, thr, prep, val, keep)
    return res, hist, curve, thr, len(Xt), len(Xv)


def run_iforest(prep_cls, train, val, seed):
    prep = prep_cls().fit(train)
    healthy = train[train["frac"] <= HEALTHY_FRAC]
    iso = IsolationForest(n_estimators=200, random_state=seed).fit(prep.transform(healthy))
    return eval_iforest(iso, prep, val)

## 4. Menjalankan eksperimen

In [ ]:
# Sel 4 - Menjalankan eksperimen (FD002 dan FD004; Strategi A dan B; LSTM-AE dan Isolation Forest)
all_rows, hist_store, curve_store, info = [], {}, {}, []
t0 = time.time()
for subset, path in DATA.items():
    df = load(path)
    train, val = split_engines(df)
    keep = sorted(val["unit"].unique())[0]          # mesin contoh untuk Gambar 4
    print(f"\n=== {subset}: {train['unit'].nunique()} mesin latih, {val['unit'].nunique()} mesin validasi ===")
    for S in (StrategyA, StrategyB):
        for seed in SEEDS:
            res, hist, curve, thr, ntr, nva = run_lstm(S, train, val, seed, keep)
            res.assign(subset=subset, model="LSTM-AE", strategy=S.name, seed=seed).pipe(all_rows.append)
            info.append(dict(subset=subset, strategy=S.name, seed=seed, tau=thr, epochs=len(hist["loss"]),
                             windows_train=ntr, windows_val=nva, val_loss=min(hist["val_loss"])))
            if seed == SEEDS[0]:
                hist_store[(subset, S.name)] = hist
                curve_store[(subset, S.name)] = curve
            res_if = run_iforest(S, train, val, seed)
            res_if.assign(subset=subset, model="Isolation Forest", strategy=S.name, seed=seed).pipe(all_rows.append)
            print(f"  {subset} strategi {S.name} seed {seed}: tau={thr:.4f}, epoch={len(hist['loss'])}, "
                  f"[{time.time() - t0:.0f} dtk]")

raw = pd.concat(all_rows, ignore_index=True)
raw.to_csv(f"{OUT}/hasil_per_mesin.csv", index=False)
pd.DataFrame(info).to_csv(f"{OUT}/info_pelatihan.csv", index=False)
print("\nSelesai dalam", round(time.time() - t0), "detik")
pd.DataFrame(info)

## 5. Tabel hasil (Tabel 4 makalah)

In [ ]:
# Sel 5 - Ringkasan: Tabel 4 makalah (rata-rata +- simpangan baku antar seed)
per_seed = raw.groupby(["subset", "model", "strategy", "seed"]).agg(
    far=("far", "mean"), recall=("recall", "mean"), lead_time=("lead_time", "mean"),
    detected=("detected", lambda s: 100 * s.mean()),
    early_false=("early_false", lambda s: 100 * s.mean()),
    lead_median=("lead_time", "median")).reset_index()
per_seed.to_csv(f"{OUT}/ringkasan_per_seed.csv", index=False)

metrics = ["far", "recall", "lead_time", "detected", "early_false"]
g = per_seed.groupby(["subset", "model", "strategy"])[metrics]
mean, std = g.mean(), g.std(ddof=0).fillna(0)
mean.to_csv(f"{OUT}/ringkasan_rata2.csv"); std.to_csv(f"{OUT}/ringkasan_std.csv")

def fmt(m, s, nd):
    return f"{m:.{nd}f} ± {s:.{nd}f}"

tabel4 = pd.DataFrame({
    "FAR": [fmt(mean.loc[i, "far"], std.loc[i, "far"], 3) for i in mean.index],
    "Recall fase akhir": [fmt(mean.loc[i, "recall"], std.loc[i, "recall"], 3) for i in mean.index],
    "Lead time (siklus)": [fmt(mean.loc[i, "lead_time"], std.loc[i, "lead_time"], 1) for i in mean.index],
    "Mesin terdeteksi (%)": [fmt(mean.loc[i, "detected"], std.loc[i, "detected"], 1) for i in mean.index],
    "Alarm dini palsu (%)": [fmt(mean.loc[i, "early_false"], std.loc[i, "early_false"], 1) for i in mean.index],
}, index=mean.index)
tabel4.to_csv(f"{OUT}/tabel4_hasil.csv")
print("Jumlah seed:", len(SEEDS), "| catatan: simpangan baku = 0 bila hanya 1 seed")
tabel4

## 6. Gambar

In [ ]:
# Sel 6 - Gambar untuk makalah (Gambar 4: skor anomali satu mesin; kurva loss)
plt.rcParams.update({"font.family": "serif", "font.size": 9})
for subset in DATA:
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharey=True)
    for ax, S in zip(axes, ("A", "B")):
        c = curve_store[(subset, S)]
        T = c["T"]
        ax.axvspan(0, HEALTHY_FRAC * T, color="#DDEEDD", alpha=.8, lw=0)
        ax.axvspan((1 - LATE_FRAC) * T, T, color="#F5D8D3", alpha=.8, lw=0)
        ax.plot(c["cycle"], c["err"], lw=.9, color="#2B3A4A")
        ax.axhline(c["thr"], color="#C0392B", ls="--", lw=1, label=r"ambang $\tau$")
        if not np.isnan(c["alarm"]):
            ax.axvline(c["alarm"], color="#E67E22", lw=1, label="alarm pertama")
        ax.set_yscale("log"); ax.set_xlabel("Siklus"); ax.set_title(f"Strategi {S}")
    axes[0].set_ylabel("Galat rekonstruksi (MSE)"); axes[0].legend(fontsize=7, loc="upper left")
    fig.tight_layout(); fig.savefig(f"{OUT}/gambar4_{subset}.png", dpi=300); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), sharey=False)
for ax, subset in zip(axes, DATA):
    for S, col in (("A", "#2E6DA4"), ("B", "#C0392B")):
        h = hist_store[(subset, S)]
        ax.plot(h["loss"], color=col, label=f"Strategi {S} (latih)")
        ax.plot(h["val_loss"], color=col, ls="--", label=f"Strategi {S} (validasi)")
    ax.set_title(subset); ax.set_xlabel("Epoch"); ax.set_yscale("log")
axes[0].set_ylabel("Loss (MSE)"); axes[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f"{OUT}/kurva_loss.png", dpi=300); plt.show()

## 7. Unduh hasil

In [ ]:
# Sel 7 - Unduh semua hasil sebagai satu file zip
import shutil
shutil.make_archive("hasil_eksperimen", "zip", OUT)
try:
    from google.colab import files
    files.download("hasil_eksperimen.zip")
except ImportError:
    print("Berkas ada di hasil_eksperimen.zip")